In [4]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine



In [24]:
def obtener_datos_ejecucion_metrica(execution_name, metric_name, engine):
    """
    Extrae todos los registros de una ejecución y métrica específica.
    """
    SQL = """
        SELECT time,instance,grupo,job,tags,label,metric_value
        FROM metricas 
        WHERE execution_name = %s 
          AND metric_name = %s
        ORDER BY time ASC
    """
    
    # Ejecutamos la consulta pasando los parámetros de forma segura
    df = pd.read_sql(SQL, engine, params=(execution_name, metric_name))
    
    # Convertimos la columna time a formato datetime
    df['time'] = pd.to_datetime(df['time'])
    df         = df.sort_values('time')


    inicio_global = df['time'].min()
    df['tiempo_relativo'] = (df['time'] - inicio_global).dt.total_seconds()
        
    return df

In [25]:
def interpolar(df_serie,INTERVALO_VALOR):
    
    # --- 1. CONFIGURACIÓN ---
    INTERVALO_VALOR = 5
    INTERVALO_MUESTREO = f'{INTERVALO_VALOR}s'
    
    # --- 2. PREPARACIÓN ---
    # Aseguramos que 'time' es datetime y ordenamos
    df_serie['time'] = pd.to_datetime(df_serie['time'])
    df_serie = df_serie.sort_values('time')
    
    # Filtramos solo las columnas que nos interesan
    cols_a_mantener = ['time', 'metric_value', 'grupo', 'instance', 'job', 'label','tags']
    cols_numericas = ['metric_value']  # Ajusta si tienes más columnas numéricas
    cols_texto = ['grupo', 'instance', 'job', 'label', 'tags']
    
    df_filtrado = df_serie[cols_a_mantener].copy()
    df_filtrado['metric_value'] = df_filtrado['metric_value'].astype(float)
    
    # --- 3. PROCESAMIENTO ---
    # Establecemos el índice temporal
    df_indexado = df_filtrado.set_index('time').sort_index()
    
    # AJUSTE: Calculamos el final redondeando al siguiente múltiplo de 5 segundos
    # Esto asegura que si tu último punto es 799, el rango llegue hasta 800
    inicio = df_indexado.index.min()
    ultimo_timestamp = df_indexado.index.max()
    # Calculamos cuánto falta para el siguiente múltiplo de 5s
    segundos_faltantes = INTERVALO_VALOR - (ultimo_timestamp.second % INTERVALO_VALOR)
    fin = df_indexado.index.max().ceil(INTERVALO_MUESTREO) + pd.Timedelta(seconds=INTERVALO_VALOR)
    
    
    rango_muestreo = pd.date_range(start=inicio, end=fin, freq=INTERVALO_MUESTREO)
    
    # Creamos un DataFrame destino con el índice regular
    df_target = pd.DataFrame(index=rango_muestreo)
    
    df_fusionado = df_target.join(df_indexado, how='outer')
    df_fusionado[cols_numericas] = df_fusionado[cols_numericas].interpolate(method='time')
    df_fusionado[cols_texto] = df_fusionado[cols_texto].ffill()
    df_final = df_fusionado.loc[df_target.index]
    inicio_global = df_final.index.min()
    df_final['tiempo_relativo'] = (df_final.index - inicio_global).total_seconds()
    return df_final


In [105]:
INTERVALO_MUESTREO = 5
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = os.getenv("SEED")

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

🔌 [OK] Conexión establecida con TimescaleDB.


In [16]:
SQL='select distinct(execution_name) from metricas'
df_metric_name = pd.read_sql(SQL, engine)
lista_execution_name=df_metric_name['execution_name'].to_list()

In [14]:
#recuperamos datos calculados previamente para realizar la transformacion del dataset original
df_ejecuciones_outlier = pd.read_parquet(f'{FOLDER_SALIDA}/df_ejecuciones_outlier.parquet')
df_analisis_metricas = pd.read_parquet(f'{FOLDER_SALIDA}/analisis_metricas.parquet')
lista_metricas_sel=df_analisis_metricas[   df_analisis_metricas['metric_name_sel']!=''  ]['metric_name'].tolist()


ruta_archivo = f'{FOLDER_SALIDA}/lista_ejecuciones_descartadas.json'
with open(ruta_archivo, "r", encoding="utf-8") as f:
    lista_ejecuciones_descartadas = json.load(f)

    



In [17]:
set_descartadas = set(lista_ejecuciones_descartadas)
lista_ejecuciones_sel = [x for x in lista_execution_name if x not in set_descartadas]

In [18]:
len(lista_ejecuciones_sel)

1732

In [22]:
una_ejecucion=lista_ejecuciones_sel[0]

In [23]:
una_ejecucion

'light-oauth2-data-1719592986.tar'

In [110]:
N_ejecucion=0
for una_ejecucion in lista_ejecuciones_sel:
    N_ejecucion+=1
    lista_df_procesados = []
    for una_metrica in lista_metricas_sel:
        df_sel = df_analisis_metricas[df_analisis_metricas['metric_name'] == una_metrica]
        clasificacion = df_sel.iloc[0]['clasificacion']
        if clasificacion=='diff':
            nombre_metrica = una_metrica+'_diff'
        else:
            nombre_metrica    = una_metrica
        print(f'execution_name = {una_ejecucion}   |  metric_name = {una_metrica}'.ljust(200),end='\r')
        df_serie        = obtener_datos_ejecucion_metrica(una_ejecucion, una_metrica, engine)
        df_procesado = (
            interpolar(df_serie, INTERVALO_MUESTREO)
            .reset_index()
            .rename(columns={'index': 'time'})
            .drop(columns=['tiempo_relativo'], errors='ignore')
        )
        df_procesado['metric_name'] = nombre_metrica
        if clasificacion == 'diff':
            df_procesado['metric_value'] = df_procesado['metric_value'].diff().fillna(0)
        lista_df_procesados.append(df_procesado)
    
    
    #obtenemos esta metrica que se vio que prsenta muchos NaN en anomalias, asi que puede servir para detectarlas
    #crearemos una metrica _isnan que sera 1 si hay valor NaN
    #modificaremos la metrica original para sustituir los NaN por valores medios
    una_metrica = 'prometheus_target_sync_length_seconds'
    df_serie    = obtener_datos_ejecucion_metrica(una_ejecucion, una_metrica, engine)
    
    
    #creamos metrica _isnan
    df_serie2  = df_serie.copy()
    df_serie2['metric_value'] = df_serie2['metric_value'].isna().astype(int)
    df_procesado = (
        interpolar(df_serie2, INTERVALO_MUESTREO)
        .reset_index()
        .rename(columns={'index': 'time'})
        .drop(columns=['tiempo_relativo'], errors='ignore')
    )
    df_procesado['metric_name'] = nombre_metrica+'_isnan'
    lista_df_procesados.append(df_procesado)
    
    
    mediana = df_serie['metric_value'].median()
    
    # Rellenamos los NaN con ese valor
    df_serie['metric_value'] = df_serie['metric_value'].fillna(mediana)
    
    
    #creamos la metrica _mod rellenando con la media
    df_procesado = (
        interpolar(df_serie, INTERVALO_MUESTREO)
        .reset_index()
        .rename(columns={'index': 'time'})
        .drop(columns=['tiempo_relativo'], errors='ignore')
    )
    df_procesado['metric_name'] = nombre_metrica+'_mod'
    lista_df_procesados.append(df_procesado)
    df_final = pd.concat(lista_df_procesados, ignore_index=True)
    
    df_final = df_final.sort_values(by=['time', 'metric_name', 'instance', 'grupo'])
    fichero_parquet=f'{FOLDER_TMP}/df_transformado({una_ejecucion}).parquet'
    df_final.to_parquet(fichero_parquet, index=False,engine='fastparquet')
    
    print(f'Exportado datos a fichero ({fichero_parquet})    N_ejecucion = {N_ejecucion:4d}',end='\r')

Exportado datos a fichero (./tmp/df_transformado(light-oauth2-data-1719592986.tar).parquet)    N_ejecucion =    1                                                                                       
Exportado datos a fichero (./tmp/df_transformado(light-oauth2-data-1719594842.tar).parquet)    N_ejecucion =    2                                                                                       
Exportado datos a fichero (./tmp/df_transformado(light-oauth2-data-1719598443.tar).parquet)    N_ejecucion =    3                                                                                       
Exportado datos a fichero (./tmp/df_transformado(light-oauth2-data-1719602043.tar).parquet)    N_ejecucion =    4                                                                                       
Exportado datos a fichero (./tmp/df_transformado(light-oauth2-data-1719605644.tar).parquet)    N_ejecucion =    5                                                                                   

./tmp/df_transformado(light-oauth2-data-1719592986.tar).parquet
